# Córdoba — Geographic Generalization and Few-Shot Adaptation

**Environment**: Colab A100 (GPU required)

**Part 1**: Zero-shot generalization — base model (trained on Corrientes) evaluated on Córdoba
**Part 2**: Few-shot domain adaptation — decoder fine-tuned on 100 Córdoba patches

In [ ]:
# [01] Environment setup: detect Colab, mount Drive, install dependencies
try:
    import google.colab
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    print('Running in Google Colab — Drive mounted.')

    needs = False
    try:
        import numpy as np
        assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0)
        from terratorch.registry import BACKBONE_REGISTRY
        import segmentation_models_pytorch
        print(f'OK — numpy={np.__version__}, terratorch ready.')
    except Exception as e:
        needs = True
        print(f'Installing ({e})...')

    if needs:
        import subprocess, sys, os
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'numpy==2.0.2', '--force-reinstall'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'terratorch', 'einops', 'timm',
                        'segmentation-models-pytorch'], check=True)
        print('Restarting kernel to pick up new numpy...')
        os.kill(os.getpid(), 9)

except ImportError:
    IN_COLAB = False
    print('Running locally.')


In [ ]:
import os, random, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

# ── Paths ─────────────────────────────────────────────────────────────────────
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
else:
    BASE = Path('G:/Mon Drive/GeoAI/wildfire-spread')

CKPT_DIR   = BASE / 'models'
RESULTS    = BASE / 'results'
RESULTS.mkdir(exist_ok=True)

# Patches de Córdoba — copiados desde local a Drive en el paso anterior
if IN_COLAB:
    LOCAL_CORDOBA = Path('/content/cordoba_patches')
    if not (LOCAL_CORDOBA / 'images').exists():
        print('Copiando patches Córdoba a /content/...')
        import subprocess
        LOCAL_CORDOBA.mkdir(exist_ok=True)
        subprocess.run(['cp', '-r', str(BASE / 'data/cordoba/patches/images'),     str(LOCAL_CORDOBA)], check=True)
        subprocess.run(['cp', '-r', str(BASE / 'data/cordoba/patches/masks_dnbr'), str(LOCAL_CORDOBA)], check=True)
        print('Copy complete.')
    IMG_DIR  = LOCAL_CORDOBA / 'images'
    MASK_DIR = LOCAL_CORDOBA / 'masks_dnbr'
else:
    IMG_DIR  = BASE / 'data/cordoba/patches/images'
    MASK_DIR = BASE / 'data/cordoba/patches/masks_dnbr'

img_paths  = sorted(IMG_DIR.glob('*.npy'))
mask_paths = sorted(MASK_DIR.glob('*.npy'))
assert len(img_paths) == len(mask_paths), f'{len(img_paths)} imgs vs {len(mask_paths)} masks'
print(f'\nPatches Córdoba: {len(img_paths)}')
fire_flags = np.array([np.load(f, mmap_mode='r').max() > 0 for f in mask_paths])
print(f'Patches con cicatriz: {fire_flags.sum()} ({fire_flags.mean()*100:.1f}%)')

In [ ]:
# ── Architecture — identical to notebook 04 (v1.5) ───────────────────────────────
from terratorch.registry import BACKBONE_REGISTRY

PRITHVI_MEAN = np.array([0.033349, 0.057012, 0.058897, 0.232325, 0.197285, 0.119449], dtype=np.float32)
PRITHVI_STD  = np.array([0.022691, 0.026808, 0.040041, 0.077917, 0.087087, 0.072420], dtype=np.float32)
BAND_IDX     = [0, 1, 2, 4, 5, 6]   # B02, B03, B04, B8A, B11, B12
IMG_SIZE     = 224
T            = (256 - IMG_SIZE) // 2
THRESHOLD    = 0.525

class MultiScaleNeck(nn.Module):
    def __init__(self, n_side=14, d_model=768, fpn_ch=256):
        super().__init__()
        self.n_side = n_side
        self.lateral = nn.ModuleList([
            nn.Sequential(nn.Conv2d(d_model, fpn_ch, 1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(4)
        ])
        self.td_conv = nn.ModuleList([
            nn.Sequential(nn.Conv2d(fpn_ch, fpn_ch, 3, padding=1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(3)
        ])
    def tok2map(self, tokens):
        B, L, D = tokens.shape
        return tokens.permute(0, 2, 1).reshape(B, D, self.n_side, self.n_side)
    def forward(self, layers_out):
        feats = [self.lateral[i](self.tok2map(layers_out[idx][:, 1:, :]))
                 for i, idx in enumerate([2, 5, 8, 11])]
        out = feats[3]
        out = self.td_conv[0](feats[2] + out)
        out = self.td_conv[1](feats[1] + out)
        out = self.td_conv[2](feats[0] + out)
        return out  # (B, 256, 14, 14)


class FPNDecoder(nn.Module):
    def __init__(self, in_ch=256):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)


class PrithviSegModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BACKBONE_REGISTRY.build('prithvi_eo_v1_100')
        self.neck     = MultiScaleNeck(n_side=14, d_model=768, fpn_ch=256)
        self.decoder  = FPNDecoder(in_ch=256)
    def forward(self, x):
        return self.decoder(self.neck(self.backbone(x.unsqueeze(2))))

print('Loading model v1.5...')
model = PrithviSegModelV2().to(DEVICE)
ckpt_path = CKPT_DIR / 'best_prithvi_burn_scar_wildfire.pth'
if not ckpt_path.exists():
    candidates = list(CKPT_DIR.glob('*prithvi*.pth'))
    if candidates:
        ckpt_path = candidates[0]
    else:
        raise FileNotFoundError(f'No checkpoint found in {CKPT_DIR}')
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
model.eval()
print(f'Model loaded: {ckpt_path.name}')

In [ ]:
# ── Dataset de Córdoba ────────────────────────────────────────────────────────
class CordobaDataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        img  = np.load(self.img_paths[idx]).astype(np.float32)    # (11, 256, 256)
        mask = np.load(self.mask_paths[idx]).astype(np.float32)   # (256, 256)
        img  = img[BAND_IDX]               # (6, 256, 256)
        img /= 10000.0
        img  = (img - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        t    = (256 - IMG_SIZE) // 2
        img  = img[:, t:t+IMG_SIZE, t:t+IMG_SIZE]
        mask = mask[t:t+IMG_SIZE, t:t+IMG_SIZE]
        return torch.from_numpy(img), torch.from_numpy(mask).unsqueeze(0)

ds     = CordobaDataset(img_paths, mask_paths)
loader = DataLoader(ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
print(f'Batches a evaluar: {len(loader)}')

In [ ]:
# ── Inferencia completa ───────────────────────────────────────────────────────
all_probs, all_targets = [], []
patch_probs_list, patch_masks_list = [], []   # no imgs — cargamos del disco al visualizar

print('Evaluando en Córdoba...')
with torch.no_grad():
    for imgs, masks in tqdm(loader):
        imgs = imgs.to(DEVICE)
        probs = model(imgs).sigmoid().squeeze(1).cpu().numpy()
        masks_np = masks.squeeze(1).cpu().numpy()

        all_probs.append(probs.reshape(-1))
        all_targets.append(masks_np.reshape(-1))

        for b in range(len(imgs)):
            patch_probs_list.append(probs[b])
            patch_masks_list.append(masks_np[b])

all_probs   = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets).astype(np.int32)
all_preds   = (all_probs > THRESHOLD).astype(np.int32)

# ── Metricas ──────────────────────────────────────────────────────────────────
precision, recall, f1, _ = precision_recall_fscore_support(
    all_targets, all_preds, pos_label=1, average='binary', zero_division=0)
tp = int(((all_preds==1)&(all_targets==1)).sum())
fp = int(((all_preds==1)&(all_targets==0)).sum())
fn = int(((all_preds==0)&(all_targets==1)).sum())
tn = int(((all_preds==0)&(all_targets==0)).sum())
iou_fire = tp / (tp + fp + fn + 1e-6)

try:
    auc = roc_auc_score(all_targets, all_probs)
except Exception:
    auc = float('nan')

print('\n' + '='*55)
print('  RESULTADOS — CORDOBA (Test geografico)')
print('='*55)
print(f'  Precision (cicatriz): {precision:.4f}')
print(f'  Recall (cicatriz)   : {recall:.4f}')
print(f'  F1-Score            : {f1:.4f}')
print(f'  IoU (cicatriz)      : {iou_fire:.4f}')
print(f'  AUC-ROC             : {auc:.4f}')
print('-'*55)
print('  COMPARATIVA:')
print(f'  Corrientes val IoU  : 0.4198  <- entrenamiento')
print(f'  Cordoba test IoU    : {iou_fire:.4f}  <- este notebook')
print('='*55)


In [ ]:
# ── Curva Precision-Recall ────────────────────────────────────────────────────
from sklearn.metrics import precision_recall_curve, roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

prec_c, rec_c, thr_c = precision_recall_curve(all_targets, all_probs)
axes[0].plot(rec_c, prec_c, color='crimson', lw=2)
axes[0].scatter([recall], [precision], color='black', zorder=5, s=80,
                label=f'thr=0.5  |  F1={f1:.3f}')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall — Cordoba test')
axes[0].legend(); axes[0].grid(alpha=0.3)

if not np.isnan(auc):
    fpr_c, tpr_c, _ = roc_curve(all_targets, all_probs)
    axes[1].plot(fpr_c, tpr_c, color='steelblue', lw=2, label=f'AUC={auc:.3f}')
    axes[1].plot([0,1],[0,1],'--',color='gray')
    axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC — Cordoba test')
    axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Geographic generalization evaluation — Cordoba Oct 2020', fontsize=12)
plt.tight_layout()
out = RESULTS / 'cordoba_evaluation_curves.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

# Free large arrays — no longer needed after this cell
del all_probs, all_targets, all_preds


In [ ]:
# ── Visualizacion de predicciones ────────────────────────────────────────────
def norm_band(x, p=2):
    v = x[x > 0]
    if not len(v): return np.zeros_like(x)
    lo, hi = np.percentile(v, [p, 100-p])
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

# Per-patch IoU from stored probs/masks
patch_ious = np.array([
    float(((p > THRESHOLD).astype(np.float32) * m).sum() /
          (((p > THRESHOLD).astype(np.float32) + m).clip(0,1).sum() + 1e-6))
    for p, m in zip(patch_probs_list, patch_masks_list)
])

fire_patch_idx = [i for i, f in enumerate(fire_flags) if f]
best_idx = sorted(fire_patch_idx, key=lambda i: patch_ious[i], reverse=True)[:4]

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
col_titles = ['RGB', 'Ground truth (dNBR)', 'Prediction (prob)', 'TP/FP/FN']
for ci, ct in enumerate(col_titles):
    axes[0, ci].set_title(ct, fontsize=10, pad=6)

for row, vi in enumerate(best_idx):
    # Load image from disk (avoids storing all 6,634 imgs in RAM)
    img_raw = np.load(img_paths[vi]).astype(np.float32)          # (11, 256, 256)
    img_np  = img_raw[BAND_IDX] / 10000.0
    img_np  = (img_np - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
    img_np  = img_np[:, T:T+IMG_SIZE, T:T+IMG_SIZE]              # (6, 224, 224)

    prob_np  = patch_probs_list[vi]
    mask_np  = patch_masks_list[vi]
    pred_bin = (prob_np > THRESHOLD).astype(np.float32)
    iou_v    = patch_ious[vi]

    def denorm(ch):
        return norm_band(img_np[ch] * PRITHVI_STD[ch] + PRITHVI_MEAN[ch])
    rgb = np.dstack([denorm(2), denorm(1), denorm(0)])

    err = np.zeros((*mask_np.shape, 3))
    err[(pred_bin==1)&(mask_np==1)] = [0.0, 0.8, 0.0]
    err[(pred_bin==1)&(mask_np==0)] = [1.0, 0.5, 0.0]
    err[(pred_bin==0)&(mask_np==1)] = [0.9, 0.1, 0.1]

    axes[row,0].imshow(rgb);                                axes[row,0].axis('off')
    axes[row,1].imshow(mask_np, cmap='Reds', vmin=0, vmax=1); axes[row,1].axis('off')
    axes[row,2].imshow(prob_np, cmap='hot',  vmin=0, vmax=1); axes[row,2].axis('off')
    axes[row,3].imshow(err);                                axes[row,3].axis('off')
    axes[row,0].set_ylabel(f'IoU={iou_v:.3f}', fontsize=9)

tp_p  = mpatches.Patch(color=[0,.8,0],   label='TP (detected)')
fp_p  = mpatches.Patch(color=[1,.5,0],   label='FP (false alarm)')
fn_p  = mpatches.Patch(color=[.9,.1,.1], label='FN (missed)')
fig.legend(handles=[tp_p,fp_p,fn_p], loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.01))

plt.suptitle(
    f'Cordoba Test  |  IoU={iou_fire:.4f}  |  Recall={recall:.4f}  |  Precision={precision:.4f}\n'
    f'Model trained on Corrientes (2022) — evaluated on Cordoba (2020)',
    fontsize=11, y=1.01)
plt.tight_layout()
out = RESULTS / 'cordoba_predictions.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

# Free patch lists — no longer needed
del patch_probs_list, patch_masks_list


In [ ]:
# ── Resumen final para portfolio ───────────────────────────────────────────────
print('\n' + '='*60)
print('  PORTFOLIO SUMMARY — Wildfire Burn Scar Detection')
print('='*60)
print('  Modelo: Prithvi-100M (IBM/NASA) + FPN decoder')
print('  Encoder: frozen (100M params pre-entrenados en HLS/S2)')
print('  Decoder: entrenado en Corrientes, Argentina (dic 2021–feb 2022)')
print('─'*60)
print('  TRAINING SET (Corrientes):')
print('    Área   : -59.5W/-29.0S → -56.0W/-26.5N (pastizales/humedales)')
print('    Labels : dNBR > 0.10 (cicatrices de incendio)')
print('    Patches: 5,687 × 256×256 px × 6 bandas S2')
print('─'*60)
print('  TEST SET (Córdoba) — zona NUNCA vista:')
print(f'    Área   : -65.5W/-33.0S → -62.5W/-30.5N (sierras/monte xerófilo)')
print(f'    Patches: {len(img_paths)}')
print(f'    IoU cicatriz : {iou_fire:.4f}')
print(f'    Recall       : {recall:.4f}')
print(f'    Precision    : {precision:.4f}')
print('─'*60)
print('  PIPELINE COMPLETO:')
print('    S2 L2A (CDSE) → dNBR labels → Prithvi fine-tune → Generalización')
print('    Corrientes 2022 → Córdoba 2020 = diferente región + diferente año')
print('='*60)

---

## Part 2 — Few-Shot Decoder Fine-Tuning (100 Córdoba Patches)

In [ ]:
# [09] Part 2 setup: constants, imports, WildfireDataset (augmented for fine-tuning)
from sklearn.model_selection import train_test_split
from segmentation_models_pytorch.losses import DiceLoss, FocalLoss

N_FINETUNE = 100
BATCH_SIZE = 8
LR_FT      = 1e-4
EPOCHS_FT  = 30
CKPT_FT    = CKPT_DIR / 'best_prithvi_burn_scar_wildfire_ft_cordoba.pth'


class WildfireDataset(Dataset):
    def __init__(self, img_paths, mask_paths, augment=False):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.augment    = augment

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        img  = np.load(self.img_paths[idx]).astype(np.float32)   # (11, 256, 256)
        mask = np.load(self.mask_paths[idx]).astype(np.float32)  # (256, 256)
        img  = img[BAND_IDX] / 10000.0
        img  = (img - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        img  = img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
        mask = mask[T:T+IMG_SIZE, T:T+IMG_SIZE]
        if self.augment:
            if random.random() > 0.5:
                img = img[:, :, ::-1].copy(); mask = mask[:, ::-1].copy()
            if random.random() > 0.5:
                img = img[:, ::-1, :].copy(); mask = mask[::-1, :].copy()
        return torch.from_numpy(img), torch.from_numpy(mask).unsqueeze(0)


print(f'Part 2 ready | N_FINETUNE={N_FINETUNE} | EPOCHS={EPOCHS_FT} | LR={LR_FT}')


In [ ]:
# ── Split: 100 fine-tune patches / resto test ─────────────────────────────────
print('Scanning fire flags...')
fire_flags = np.array([
    np.load(p, mmap_mode='r').max() > 0 for p in tqdm(mask_paths)
], dtype=np.int32)

indices = np.arange(len(img_paths))

# Separar 100 patches para fine-tuning (estratificados)
ft_idx, test_idx = train_test_split(
    indices,
    train_size=N_FINETUNE,
    stratify=fire_flags,
    random_state=SEED
)

print(f'Fine-tune : {len(ft_idx):>5} patches ({fire_flags[ft_idx].sum()} positive, '
      f'{fire_flags[ft_idx].mean()*100:.1f}%)')
print(f'Test      : {len(test_idx):>5} patches ({fire_flags[test_idx].sum()} positive, '
      f'{fire_flags[test_idx].mean()*100:.1f}%)')

ft_ds   = WildfireDataset([img_paths[i] for i in ft_idx],   [mask_paths[i] for i in ft_idx],   augment=True)
test_ds = WildfireDataset([img_paths[i] for i in test_idx], [mask_paths[i] for i in test_idx], augment=False)

ft_loader   = DataLoader(ft_ds,   batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Fine-tune batches: {len(ft_loader)}  |  Test batches: {len(test_loader)}')

In [ ]:
# ── Evaluación ANTES del fine-tuning (baseline Córdoba) ──────────────────────
def evaluate(model, loader, threshold=THRESHOLD):
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc='Evaluating', leave=False):
            probs    = model(imgs.to(DEVICE)).sigmoid().squeeze(1).cpu().numpy()
            masks_np = masks.squeeze(1).cpu().numpy()
            all_probs.append(probs.reshape(-1))
            all_targets.append(masks_np.reshape(-1))
    all_probs   = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets).astype(np.int32)
    all_preds   = (all_probs > threshold).astype(np.int32)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, pos_label=1, average='binary', zero_division=0)
    tp = int(((all_preds==1)&(all_targets==1)).sum())
    fp = int(((all_preds==1)&(all_targets==0)).sum())
    fn = int(((all_preds==0)&(all_targets==1)).sum())
    iou = tp / (tp + fp + fn + 1e-6)
    try:
        auc = roc_auc_score(all_targets, all_probs)
    except Exception:
        auc = float('nan')
    return dict(iou=iou, precision=prec, recall=rec, f1=f1, auc=auc)


print('Evaluating BASE model (no fine-tuning) on Córdoba test set...')
results_before = evaluate(model, test_loader)
print(f'  IoU       : {results_before["iou"]:.4f}')
print(f'  Recall    : {results_before["recall"]:.4f}')
print(f'  Precision : {results_before["precision"]:.4f}')
print(f'  AUC-ROC   : {results_before["auc"]:.4f}')

In [ ]:
# ── Decoder fine-tuning on 100 Córdoba patches ───────────────────────────────
def criterion(pred, target):
    return DiceLoss(mode='binary', from_logits=True)(pred, target) + \
           FocalLoss(mode='binary', gamma=2.0, alpha=0.75)(pred, target)

# Freeze backbone; train only neck + decoder
for p in model.parameters():
    p.requires_grad = False
for p in model.neck.parameters():
    p.requires_grad = True
for p in model.decoder.parameters():
    p.requires_grad = True

trainable = [p for p in model.parameters() if p.requires_grad]
print(f'Trainable params: {sum(p.numel() for p in trainable):,}')

optimizer = torch.optim.AdamW(trainable, lr=LR_FT, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS_FT, eta_min=1e-7)

best_iou_ft  = 0.0
ft_losses, ft_ious = [], []

print(f'Fine-tuning decoder — {EPOCHS_FT} epochs | LR={LR_FT} | {N_FINETUNE} Córdoba patches')
for epoch in range(1, EPOCHS_FT + 1):
    model.train()
    model.backbone.eval()
    ep_loss = 0.0
    for imgs, masks in tqdm(ft_loader, desc=f'FT {epoch:02d}/{EPOCHS_FT}', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward(); optimizer.step()
        ep_loss += loss.item()
    scheduler.step()
    ft_losses.append(ep_loss / len(ft_loader))

    res = evaluate(model, test_loader)
    ft_ious.append(res['iou'])
    print(f'Epoch {epoch:02d} | Loss: {ft_losses[-1]:.4f} | IoU: {ft_ious[-1]:.4f}')

    if ft_ious[-1] > best_iou_ft:
        best_iou_ft = ft_ious[-1]
        torch.save(model.state_dict(), CKPT_FT)
        print(f'  Saved (IoU: {best_iou_ft:.4f})')

print(f'\nBest IoU fine-tuned: {best_iou_ft:.4f}')


In [ ]:
# ── Evaluación DESPUÉS del fine-tuning ───────────────────────────────────────
model.load_state_dict(torch.load(CKPT_FT, map_location=DEVICE))
results_after = evaluate(model, test_loader)

print('\n' + '='*60)
print('  COMPARISON — Córdoba test set')
print('='*60)
print(f'  {"Métrica":<12} {"Base (07)":>12} {"Fine-tuned":>12} {"Mejora":>10}')
print('-'*60)
for key in ['iou', 'recall', 'precision', 'auc']:
    label = {'iou': 'IoU', 'recall': 'Recall', 'precision': 'Precision', 'auc': 'AUC-ROC'}[key]
    before = results_before[key]
    after  = results_after[key]
    delta  = after - before
    print(f'  {label:<12} {before:>12.4f} {after:>12.4f} {delta:>+10.4f}')
print('='*60)
print(f'\nFine-tuned model saved: {CKPT_FT.name}')

In [ ]:
# ── Curvas de fine-tuning ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, EPOCHS_FT+1), ft_losses, color='steelblue', label='Fine-tune loss')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss — fine-tuning Cordoba')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(range(1, EPOCHS_FT+1), ft_ious, color='green', label='IoU Cordoba')
axes[1].axhline(results_before['iou'], color='gray', ls=':', label=f'Base: {results_before["iou"]:.4f}')
axes[1].axhline(best_iou_ft,           color='red',  ls='--', label=f'Best FT: {best_iou_ft:.4f}')
axes[1].set(xlabel='Epoch', ylabel='IoU', title='IoU Cordoba — fine-tuning')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
out = RESULTS / 'cordoba_finetune_curves.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out}')

In [ ]:
# ── Visualización — predicciones del modelo fine-tuned ───────────────────────
def norm_band(x, p=2):
    v = x[x > 0]
    if not len(v): return np.zeros_like(x)
    lo, hi = np.percentile(v, [p, 100-p])
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

model.eval()
sample_imgs, sample_masks, sample_probs = [], [], []
with torch.no_grad():
    for imgs, masks in test_loader:
        probs = model(imgs.to(DEVICE)).sigmoid().squeeze(1).cpu().numpy()
        for b in range(len(imgs)):
            m = masks[b].squeeze().numpy()
            if m.max() > 0:
                sample_imgs.append(imgs[b].numpy())
                sample_masks.append(m)
                sample_probs.append(probs[b])
        if len(sample_imgs) >= 3:
            break

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for row in range(min(3, len(sample_imgs))):
    img_np  = sample_imgs[row]
    prob_np = sample_probs[row]
    mask_np = sample_masks[row]
    pred_b  = (prob_np > THRESHOLD).astype(np.float32)

    def dn(ch):
        return norm_band(img_np[ch] * PRITHVI_STD[ch] + PRITHVI_MEAN[ch])
    rgb = np.dstack([dn(2), dn(1), dn(0)])

    err = np.zeros((*mask_np.shape, 3))
    err[(pred_b==1)&(mask_np==1)] = [0.0, 0.8, 0.0]
    err[(pred_b==1)&(mask_np==0)] = [1.0, 0.5, 0.0]
    err[(pred_b==0)&(mask_np==1)] = [0.9, 0.1, 0.1]

    axes[row,0].imshow(rgb);                                    axes[row,0].axis('off')
    axes[row,1].imshow(mask_np, cmap='Reds', vmin=0, vmax=1);  axes[row,1].axis('off')
    axes[row,2].imshow(prob_np, cmap='hot',  vmin=0, vmax=1);  axes[row,2].axis('off')
    axes[row,3].imshow(err);                                    axes[row,3].axis('off')

for ax, t in zip(axes[0], ['RGB', 'Ground truth (dNBR)', 'Prediction (prob)', 'TP/FP/FN']):
    ax.set_title(t, fontsize=10)

tp_p = mpatches.Patch(color=[0,.8,0], label='TP')
fp_p = mpatches.Patch(color=[1,.5,0], label='FP')
fn_p = mpatches.Patch(color=[.9,.1,.1], label='FN')
fig.legend(handles=[tp_p,fp_p,fn_p], loc='lower center', ncol=3, bbox_to_anchor=(0.5,-0.01))
plt.suptitle(f'Cordoba — Fine-tuned | IoU base: {results_before["iou"]:.4f} → FT: {best_iou_ft:.4f}',
             fontsize=12, y=1.01)
plt.tight_layout()
out = RESULTS / 'cordoba_finetune_predictions.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out}')

In [ ]:
# ── Resumen portfolio ─────────────────────────────────────────────────────────
print('='*65)
print('  PORTFOLIO SUMMARY — Few-Shot Domain Adaptation')
print('='*65)
print(f'  Modelo base  : Prithvi-100M + FPN (trained on Corrientes 2022)')
print(f'  Fine-tune    : {N_FINETUNE} Córdoba 2020 patches ({EPOCHS_FT} épocas)')
print(f'  Strategy     : encoder frozen, decoder only updated')
print('-'*65)
print(f'  {"Métrica":<12} {"Corrientes val":>14} {"Córdoba base":>14} {"Córdoba FT":>12}')
print('-'*65)
print(f'  {"IoU":<12} {0.5385:>14.4f} {results_before["iou"]:>14.4f} {best_iou_ft:>12.4f}')
print(f'  {"Recall":<12} {0.7068:>14.4f} {results_before["recall"]:>14.4f} {results_after["recall"]:>12.4f}')
print(f'  {"Precision":<12} {0.6934:>14.4f} {results_before["precision"]:>14.4f} {results_after["precision"]:>12.4f}')
print('='*65)

---

## Part 3 — T=2 Zero-Shot: does temporal change detection transfer to Córdoba?

The v1.6 Siamese model was trained on Corrientes T=2 pairs (pre-fire Oct-Nov 2021 + post-fire Dec 2021-Feb 2022).
Here we evaluate it zero-shot on Córdoba T=2 pairs (pre-fire Aug-Sep 2020 + post-fire Nov-Dec 2020), a region and biome never seen during training.

For a fair comparison with v1.5 ZS, both models are evaluated on the same 3,591 T=2 pairs.

In [ ]:
# [15] Part 3 setup — copy Cordoba pre-fire patches + manifest to /content/
import json, subprocess, shutil
from pathlib import Path

if IN_COLAB:
    LOCAL_T2 = Path('/content/cordoba_t2')
    LOCAL_T2.mkdir(exist_ok=True)

    for subdir in ['images', 'images_prefire', 'masks_dnbr']:
        dst = LOCAL_T2 / subdir
        src = BASE / 'data' / 'cordoba' / 'patches' / subdir
        if not dst.exists():
            print(f'Copying {subdir}...')
            subprocess.run(['cp', '-r', str(src), str(LOCAL_T2)], check=True)
        else:
            print(f'Already exists: {subdir}')

    manifest_src = BASE / 'data' / 'cordoba' / 'patches' / 't2_pairs.json'
    shutil.copy(str(manifest_src), str(LOCAL_T2 / 't2_pairs.json'))

    T2_IMG_DIR  = LOCAL_T2 / 'images'
    T2_PRE_DIR  = LOCAL_T2 / 'images_prefire'
    T2_MASK_DIR = LOCAL_T2 / 'masks_dnbr'
    T2_MANIFEST = LOCAL_T2 / 't2_pairs.json'
else:
    T2_IMG_DIR  = BASE / 'data/cordoba/patches/images'
    T2_PRE_DIR  = BASE / 'data/cordoba/patches/images_prefire'
    T2_MASK_DIR = BASE / 'data/cordoba/patches/masks_dnbr'
    T2_MANIFEST = BASE / 'data/cordoba/patches/t2_pairs.json'

with open(T2_MANIFEST) as f:
    t2_pairs = json.load(f)

print(f'T=2 Cordoba pairs: {len(t2_pairs)}')
fire_flags_t2 = np.array([np.load(T2_MASK_DIR / n, mmap_mode='r').max() > 0
                           for n in t2_pairs], dtype=np.int32)
print(f'Positive patches : {fire_flags_t2.sum()} ({fire_flags_t2.mean()*100:.1f}%)')

In [ ]:
# [16] T=2 model architecture (identical to 04b_prithvi_t2.ipynb) + load checkpoint
class TemporalFusionNeck(nn.Module):
    def __init__(self, n_side=14, d_model=768, fpn_ch=256):
        super().__init__()
        self.n_side = n_side
        self.t_fuse = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(2 * d_model, d_model, 1, bias=False),
                nn.BatchNorm2d(d_model), nn.GELU()
            ) for _ in range(4)
        ])
        self.lateral = nn.ModuleList([
            nn.Sequential(nn.Conv2d(d_model, fpn_ch, 1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(4)
        ])
        self.td_conv = nn.ModuleList([
            nn.Sequential(nn.Conv2d(fpn_ch, fpn_ch, 3, padding=1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(3)
        ])

    def tok2map(self, tokens):
        B, L, D = tokens.shape
        return tokens.permute(0, 2, 1).reshape(B, D, self.n_side, self.n_side)

    def forward(self, pre_layers, post_layers):
        feats = []
        for i, idx in enumerate([2, 5, 8, 11]):
            pre_map  = self.tok2map(pre_layers[idx][:, 1:, :])
            post_map = self.tok2map(post_layers[idx][:, 1:, :])
            fused    = self.t_fuse[i](torch.cat([pre_map, post_map], dim=1))
            feats.append(self.lateral[i](fused))
        out = feats[3]
        out = self.td_conv[0](feats[2] + out)
        out = self.td_conv[1](feats[1] + out)
        out = self.td_conv[2](feats[0] + out)
        return out


class FPNDecoderT2(nn.Module):
    def __init__(self, in_ch=256):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)


class PrithviT2Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BACKBONE_REGISTRY.build('prithvi_eo_v1_100')
        self.neck     = TemporalFusionNeck(n_side=14, d_model=768, fpn_ch=256)
        self.decoder  = FPNDecoderT2(in_ch=256)

    def forward(self, pre, post):
        pre_layers  = self.backbone(pre.unsqueeze(2))
        post_layers = self.backbone(post.unsqueeze(2))
        return self.decoder(self.neck(pre_layers, post_layers))

THRESHOLD_T2 = 0.450   # optimal threshold from v1.6 Corrientes training

print('Loading v1.6 T=2 checkpoint...')
model_t2 = PrithviT2Model().to(DEVICE)
ckpt_t2  = CKPT_DIR / 'best_prithvi_burn_scar_t2.pth'
model_t2.load_state_dict(torch.load(ckpt_t2, map_location=DEVICE))
model_t2.eval()
print(f'Loaded: {ckpt_t2.name}')

In [ ]:
# [17] Cordoba T=2 datasets + loaders
class CordobaT2Dataset(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self): return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        raw_post = np.load(T2_IMG_DIR / name, mmap_mode='r').astype(np.float32)
        raw_pre  = np.load(T2_PRE_DIR / name, mmap_mode='r').astype(np.float32)
        mask     = np.load(T2_MASK_DIR / name, mmap_mode='r')

        def normalize(raw):
            img = (raw[BAND_IDX] / 10000.0 - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
            return img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]

        post_t = torch.from_numpy(normalize(raw_post))
        pre_t  = torch.from_numpy(normalize(raw_pre))
        mask_c = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy().astype(np.float32)
        return pre_t, post_t, torch.from_numpy(mask_c).unsqueeze(0)


class CordobaST2Dataset(Dataset):
    """Single-temporal subset using only post-fire patches from the T=2 pairs (for fair v1.5 comparison)."""
    def __init__(self, names):
        self.names = names

    def __len__(self): return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        raw_post = np.load(T2_IMG_DIR / name, mmap_mode='r').astype(np.float32)
        mask     = np.load(T2_MASK_DIR / name, mmap_mode='r')
        img = (raw_post[BAND_IDX] / 10000.0 - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        img = img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
        mask_c = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy().astype(np.float32)
        return torch.from_numpy(img), torch.from_numpy(mask_c).unsqueeze(0)


t2_loader_full  = DataLoader(CordobaT2Dataset(t2_pairs),  batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
st2_loader_full = DataLoader(CordobaST2Dataset(t2_pairs), batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f'T=2 loader (v1.6) : {len(t2_loader_full)} batches  ({len(t2_pairs)} patches)')
print(f'T=1 loader (v1.5) : {len(st2_loader_full)} batches  (same patches, post-fire only)')

In [ ]:
# [18] Zero-shot inference: v1.5 (T=1) vs v1.6 (T=2) on same Cordoba T=2 subset

# Reload base v1.5 checkpoint — Part 2 fine-tuning modifies model in-place
for p in model.parameters():
    p.requires_grad = True   # unfreeze so load_state_dict works cleanly
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
model.eval()
print(f'v1.5 base checkpoint reloaded: {ckpt_path.name}')


def _metrics(probs, targets, threshold):
    preds = (probs > threshold).astype(np.int32)
    prec, rec, f1, _ = precision_recall_fscore_support(
        targets, preds, pos_label=1, average='binary', zero_division=0)
    tp = int(((preds==1)&(targets==1)).sum())
    fp = int(((preds==1)&(targets==0)).sum())
    fn = int(((preds==0)&(targets==1)).sum())
    iou = tp / (tp + fp + fn + 1e-6)
    try:
        auc = roc_auc_score(targets, probs)
    except Exception:
        auc = float('nan')
    return dict(iou=iou, precision=prec, recall=rec, f1=f1, auc=auc)


def eval_t1_zs(loader, threshold=THRESHOLD):
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc='v1.5 ZS (T=1 subset)', leave=False):
            probs = model(imgs.to(DEVICE)).sigmoid().squeeze(1).cpu().numpy()
            all_probs.append(probs.ravel())
            all_targets.append(masks.squeeze(1).cpu().numpy().ravel())
    return _metrics(np.concatenate(all_probs), np.concatenate(all_targets).astype(np.int32), threshold)


def eval_t2_zs(loader, threshold=THRESHOLD_T2):
    model_t2.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for pre, post, masks in tqdm(loader, desc='v1.6 ZS (T=2)', leave=False):
            probs = model_t2(pre.to(DEVICE), post.to(DEVICE)).sigmoid().squeeze(1).cpu().numpy()
            all_probs.append(probs.ravel())
            all_targets.append(masks.squeeze(1).cpu().numpy().ravel())
    return _metrics(np.concatenate(all_probs), np.concatenate(all_targets).astype(np.int32), threshold)


print('Running zero-shot evaluations on the 3,591 Cordoba T=2 pairs...')
zs_t1 = eval_t1_zs(st2_loader_full)
zs_t2 = eval_t2_zs(t2_loader_full)

print()
print('=' * 62)
print('  ZERO-SHOT COMPARISON  (n=3,591, same subset)')
print('=' * 62)
print(f'  {"Metric":<12} {"v1.5 ZS (T=1)":>16} {"v1.6 ZS (T=2)":>16} {"Delta":>10}')
print('-' * 62)
for key, label in [('iou','IoU'), ('recall','Recall'), ('precision','Precision'), ('auc','AUC-ROC')]:
    before = zs_t1[key]; after = zs_t2[key]
    print(f'  {label:<12} {before:>16.4f} {after:>16.4f} {after-before:>+10.4f}')
print('=' * 62)


In [ ]:
# [19] Comparison figure: v1.5 ZS vs v1.6 T=2 ZS on Cordoba
fig, axes = plt.subplots(1, 2, figsize=(11, 5), facecolor='white')

labels  = ['IoU', 'Recall', 'Precision', 'AUC-ROC']
vals_t1 = [zs_t1['iou'], zs_t1['recall'], zs_t1['precision'], zs_t1['auc']]
vals_t2 = [zs_t2['iou'], zs_t2['recall'], zs_t2['precision'], zs_t2['auc']]
x, w    = np.arange(4), 0.32

b1 = axes[0].bar(x - w/2, vals_t1, w, label='v1.5 ZS (T=1, post-fire only)',
                 color='#aec7e8', edgecolor='none')
b2 = axes[0].bar(x + w/2, vals_t2, w, label='v1.6 ZS (T=2, pre+post)',
                 color='#1f5fa6', edgecolor='none')
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.012,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_ylim(0, 1.0); axes[0].set_ylabel('Score')
axes[0].legend(fontsize=9, frameon=False)
axes[0].spines[['top','right']].set_visible(False)
axes[0].yaxis.grid(True, alpha=0.25)
axes[0].set_title('Zero-shot transfer to Cordoba\n(n=3,591 T=2 pairs, same subset)', fontsize=10)

deltas = [zs_t2[k] - zs_t1[k] for k in ['iou', 'recall', 'precision', 'auc']]
colors = ['#2ca02c' if d >= 0 else '#d62728' for d in deltas]
axes[1].bar(labels, deltas, color=colors, edgecolor='none')
for i, d in enumerate(deltas):
    axes[1].text(i, d + 0.003 * (1 if d >= 0 else -1),
                 f'{d:+.3f}', ha='center', va='bottom' if d >= 0 else 'top',
                 fontsize=10, fontweight='bold')
axes[1].axhline(0, color='#333', lw=1.0)
axes[1].set_ylabel('Delta (T=2 minus T=1)')
axes[1].set_title('Improvement from temporal fusion\n(v1.6 T=2 vs v1.5 T=1)', fontsize=10)
axes[1].spines[['top','right']].set_visible(False)
axes[1].yaxis.grid(True, alpha=0.25)

plt.suptitle('Geographic generalization — Cordoba zero-shot  |  Prithvi-EO-1.0-100M',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
out = RESULTS / 'cordoba_t2_zs_comparison.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out}')

---

## Part 4 — T=2 Few-Shot Fine-Tuning (100 Córdoba pairs)

Same protocol as Part 2 but using T=2 paired inputs. The FPN decoder and TemporalFusionNeck are fine-tuned on 100 Córdoba pairs; the Prithvi backbone remains frozen throughout. The remaining 3,491 pairs are held out as the test set.

In [ ]:
# [20] Part 4 setup — split T=2 pairs: 100 fine-tune / rest test
from sklearn.model_selection import train_test_split as tts
import segmentation_models_pytorch as smp

N_FT_T2    = 100
EPOCHS_T2  = 30
LR_T2      = 1e-4
CKPT_T2_FT = CKPT_DIR / 'best_prithvi_burn_scar_t2_cordoba_ft.pth'

t2_idx = np.arange(len(t2_pairs))
ft_idx_t2, test_idx_t2 = tts(
    t2_idx, train_size=N_FT_T2, stratify=fire_flags_t2, random_state=SEED)

ft_names_t2   = [t2_pairs[i] for i in ft_idx_t2]
test_names_t2 = [t2_pairs[i] for i in test_idx_t2]

ft_loader_t2   = DataLoader(CordobaT2Dataset(ft_names_t2),   batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
test_loader_t2 = DataLoader(CordobaT2Dataset(test_names_t2), batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f'T=2 fine-tune set : {len(ft_names_t2)} pairs')
print(f'T=2 test set      : {len(test_names_t2)} pairs')

zs_t2_test = eval_t2_zs(test_loader_t2)
print(f'\nZero-shot baseline on test subset: IoU={zs_t2_test["iou"]:.4f}  AUC={zs_t2_test["auc"]:.4f}')

In [ ]:
# [21] T=2 decoder fine-tuning on 100 Cordoba pairs (encoder frozen throughout)
dice_loss_t2  = smp.losses.DiceLoss(mode='binary', from_logits=True)
focal_loss_t2 = smp.losses.FocalLoss(mode='binary', gamma=2.0, alpha=0.75)

def criterion_t2(pred, target):
    return dice_loss_t2(pred, target) + focal_loss_t2(pred, target)

for p in model_t2.parameters():
    p.requires_grad = False
for p in model_t2.neck.parameters():
    p.requires_grad = True
for p in model_t2.decoder.parameters():
    p.requires_grad = True

optimizer_t2 = torch.optim.AdamW(
    [p for p in model_t2.parameters() if p.requires_grad], lr=LR_T2, weight_decay=1e-4)
scheduler_t2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_t2, T_max=EPOCHS_T2, eta_min=1e-7)

best_iou_t2 = 0.0
ft_losses_t2, ft_ious_t2 = [], []

print(f'Fine-tuning T=2 neck + decoder — {EPOCHS_T2} epochs | LR={LR_T2} | backbone frozen')
for epoch in range(1, EPOCHS_T2 + 1):
    model_t2.train()
    model_t2.backbone.eval()
    ep_loss = 0.0
    for pre, post, masks in tqdm(ft_loader_t2, desc=f'T2-FT {epoch:02d}/{EPOCHS_T2}', leave=False):
        pre, post, masks = pre.to(DEVICE), post.to(DEVICE), masks.to(DEVICE)
        optimizer_t2.zero_grad()
        loss = criterion_t2(model_t2(pre, post), masks)
        loss.backward()
        optimizer_t2.step()
        ep_loss += loss.item()
    scheduler_t2.step()
    ft_losses_t2.append(ep_loss / len(ft_loader_t2))

    res = eval_t2_zs(test_loader_t2)
    ft_ious_t2.append(res['iou'])
    print(f'Epoch {epoch:02d} | Loss: {ft_losses_t2[-1]:.4f} | IoU: {ft_ious_t2[-1]:.4f}')

    if ft_ious_t2[-1] > best_iou_t2:
        best_iou_t2 = ft_ious_t2[-1]
        torch.save(model_t2.state_dict(), CKPT_T2_FT)
        print(f'  Saved (IoU: {best_iou_t2:.4f})')

print(f'\nBest T=2 FT IoU: {best_iou_t2:.4f}')

In [ ]:
# [22] Final summary: all 4 conditions + training curves
model_t2.load_state_dict(torch.load(CKPT_T2_FT, map_location=DEVICE))
ft_t2 = eval_t2_zs(test_loader_t2)

print()
print('=' * 72)
print('  FINAL SUMMARY — Cordoba geographic generalization')
print('=' * 72)
print(f'  {"Condition":<30} {"IoU":>8} {"Recall":>8} {"Precision":>10} {"AUC-ROC":>9}')
print('-' * 72)
for label, r in [
    ('v1.5 ZS T=1  (all 6,634)',  zs_t1),
    ('v1.6 ZS T=2  (3,591 pairs)', zs_t2_test),
    ('v1.5 FT T=1  (100 patches)', results_after),
    ('v1.6 FT T=2  (100 pairs)',   ft_t2),
]:
    print(f'  {label:<30} {r["iou"]:>8.4f} {r["recall"]:>8.4f} {r["precision"]:>10.4f} {r["auc"]:>9.4f}')
print('=' * 72)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), facecolor='white')
axes[0].plot(range(1, EPOCHS_T2+1), ft_losses_t2, color='steelblue')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss — T=2 fine-tuning (Cordoba)')
axes[0].grid(alpha=0.3)

axes[1].plot(range(1, EPOCHS_T2+1), ft_ious_t2, color='green', label='T=2 FT IoU')
axes[1].axhline(zs_t2_test['iou'], color='gray', ls=':', label=f'T=2 ZS: {zs_t2_test["iou"]:.4f}')
axes[1].axhline(best_iou_t2,        color='red',  ls='--', label=f'Best FT: {best_iou_t2:.4f}')
axes[1].set(xlabel='Epoch', ylabel='IoU', title='IoU Cordoba — T=2 fine-tuning')
axes[1].legend(frameon=False); axes[1].grid(alpha=0.3)

plt.tight_layout()
out = RESULTS / 'cordoba_t2_finetune_curves.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out}')

In [ ]:
# [23] Geographic evaluation summary figure — all results in one panel
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Mount Drive and resolve paths
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    BASE    = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
    RESULTS = BASE / 'results'
    if not RESULTS.exists():
        raise FileNotFoundError(f'results/ not found at {RESULTS}. Check Drive path.')
    print(f'Drive mounted. RESULTS = {RESULTS}')
except ImportError:
    BASE    = Path('G:/Mon Drive/GeoAI/wildfire-spread')
    RESULTS = BASE / 'results'
    print(f'Running locally. RESULTS = {RESULTS}')

print(f'results/ exists: {RESULTS.exists()}')
print(f'Files in results/: {len(list(RESULTS.glob("*.png")))} PNG files')

# Corrientes val IoU progression (threshold-optimized, confirmed in validation_overview_v15.png)
VERSIONS = ['v1.0', 'v1.2', 'v1.4', 'v1.5', 'v1.6']
IOU_CORR = [0.013,  0.220,  0.380,  0.538,  0.639]

# Cordoba results — run 2026-07-05
zs_t1         = dict(iou=0.1149, recall=0.2089, precision=0.2033, auc=0.7383)
zs_t2         = dict(iou=0.0870, recall=0.1617, precision=0.1584, auc=0.6157)
results_after = dict(iou=0.3289, recall=0.5399, precision=0.4570, auc=0.8695)
ft_t2         = dict(iou=0.8095, recall=0.8886, precision=0.9009, auc=0.9935)

cordoba_conditions = [
    ('v1.5 ZS\n(T=1)',         zs_t1),
    ('v1.6 ZS\n(T=2)',         zs_t2),
    ('v1.5 FT\n(100 patches)', results_after),
    ('v1.6 FT\n(T=2, 100p)',  ft_t2),
]
cond_labels = [c[0] for c in cordoba_conditions]
cond_iou    = [c[1]['iou']       for c in cordoba_conditions]
cond_rec    = [c[1]['recall']    for c in cordoba_conditions]
cond_prec   = [c[1]['precision'] for c in cordoba_conditions]

fig = plt.figure(figsize=(16, 5), facecolor='white')
fig.suptitle(
    'Wildfire Burn Scar Detection — Prithvi-EO-1.0-100M\n'
    'Training: Corrientes 2022   |   Test: Cordoba 2020 (unseen region)',
    fontsize=12, fontweight='bold', y=1.01)

# Panel 1: Model progression
ax1 = fig.add_subplot(1, 3, 1)
colors_prog = ['#c7d9f0'] * (len(VERSIONS) - 1) + ['#1f5fa6']
bars = ax1.bar(VERSIONS, IOU_CORR, color=colors_prog, edgecolor='none', width=0.6)
for bar, val in zip(bars, IOU_CORR):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.015,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.set_ylim(0, 0.85)
ax1.set_ylabel('IoU (burn scar class)')
ax1.set_title('Model Progression\n(Corrientes val set)', fontsize=10)
ax1.spines[['top', 'right']].set_visible(False)
ax1.yaxis.grid(True, alpha=0.25, zorder=0)
ax1.set_axisbelow(True)

# Panel 2: Cordoba ZS+FT grouped bar
ax2 = fig.add_subplot(1, 3, 2)
x, w = np.arange(4), 0.28
b_iou  = ax2.bar(x - w, cond_iou,  w, label='IoU',       color='#1f5fa6', edgecolor='none')
b_rec  = ax2.bar(x,     cond_rec,  w, label='Recall',    color='#aec7e8', edgecolor='none')
b_prec = ax2.bar(x + w, cond_prec, w, label='Precision', color='#6baed6', edgecolor='none')
for bars_group in (b_iou, b_rec, b_prec):
    for bar in bars_group:
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2, h + 0.008,
                 f'{h:.2f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(cond_labels, fontsize=8.5)
ax2.set_ylim(0, 1.05)
ax2.set_ylabel('Score')
ax2.set_title('Cordoba Evaluation\n(ZS and few-shot FT)', fontsize=10)
ax2.legend(fontsize=8, frameon=False, loc='upper left')
ax2.spines[['top', 'right']].set_visible(False)
ax2.yaxis.grid(True, alpha=0.25, zorder=0)
ax2.set_axisbelow(True)

# Panel 3: delta T=2 vs T=1 — red=ZS (losses), green=FT (gains)
ax3 = fig.add_subplot(1, 3, 3)
metrics  = ['IoU', 'Recall', 'Precision', 'AUC-ROC']
keys     = ['iou', 'recall', 'precision', 'auc']
delta_zs = [zs_t2[k] - zs_t1[k] for k in keys]
delta_ft = [ft_t2[k] - results_after[k] for k in keys]
xx = np.arange(4)
bz = ax3.bar(xx - 0.2, delta_zs, 0.38,
             label='T=2 ZS vs T=1 ZS', color='#d62728', alpha=0.85, edgecolor='none')
bf = ax3.bar(xx + 0.2, delta_ft, 0.38,
             label='T=2 FT vs T=1 FT', color='#2ca02c', alpha=0.85, edgecolor='none')
for bgroup in (bz, bf):
    for bar in bgroup:
        h = bar.get_height()
        sign = 1 if h >= 0 else -1
        ax3.text(bar.get_x() + bar.get_width()/2,
                 h + sign * 0.004,
                 f'{h:+.3f}', ha='center',
                 va='bottom' if h >= 0 else 'top',
                 fontsize=8, fontweight='bold')
ax3.axhline(0, color='#333', lw=0.8)
ax3.set_xticks(xx); ax3.set_xticklabels(metrics, fontsize=9)
ax3.set_ylabel('Delta (T=2 minus T=1)')
ax3.set_title('Gain from Temporal Fusion\n(T=2 Siamese vs T=1 baseline)', fontsize=10)
ax3.legend(fontsize=8, frameon=False)
ax3.spines[['top', 'right']].set_visible(False)
ax3.yaxis.grid(True, alpha=0.25, zorder=0)
ax3.set_axisbelow(True)

plt.tight_layout()
out = RESULTS / 'cordoba_evaluation_overview.png'
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out}')
